# Module 5.1: Signal Generation Pipeline

## Learning Objectives
By completing this notebook, you will:
1. Design signal generation frameworks for commodity trading
2. Convert LLM analysis into discrete trading signals
3. Implement confidence-weighted position sizing
4. Build end-to-end signal pipelines with monitoring

## Prerequisites
- Completion of Module 4 (Fundamentals Modeling)
- Understanding of trading signal concepts
- Familiarity with risk management basics

## Estimated Time: 90 minutes

---

## Setup & Imports

In [ ]:
# Purpose: Import libraries for signal generation
# Key Concept: Trading signal infrastructure and LLM integration

import os
import json
from datetime import datetime, timedelta
from typing import Dict, List, Optional, Literal
from dataclasses import dataclass, asdict, field
from enum import Enum
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from langchain_anthropic import ChatAnthropic
from langchain.prompts import ChatPromptTemplate
from langchain.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field, validator

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
np.random.seed(42)

print("✓ Imports complete")

## 1. Understanding Trading Signals

### What is a Trading Signal?

A **trading signal** is a structured recommendation to take a position based on analysis:

```
Signal Components:
├── Direction: Long / Short / Neutral
├── Confidence: 0-100% conviction level
├── Time Horizon: Days to weeks the signal is valid
├── Rationale: Why this signal was generated
├── Sources: Data/analysis inputs
└── Metadata: Additional context
```

### LLM Signal Generation Workflow

```
Data Sources → LLM Analysis → Signal Logic → Position Size → Risk Check
     ↓              ↓             ↓            ↓              ↓
  Reports      Sentiment     Threshold    Confidence     Limits
  Prices       Fundamentals  Rules        Based          Applied
  News         Forecasts                  Sizing
```

### Signal Quality Metrics

1. **Precision**: % of profitable signals (hit rate)
2. **Profit Factor**: Avg win / avg loss
3. **Sharpe Ratio**: Risk-adjusted returns
4. **Confidence Calibration**: Does 70% confidence = 70% win rate?

## 2. Signal Data Model

In [ ]:
# Purpose: Define structured signal data model
# Key Concept: Type-safe signal representation

class SignalDirection(str, Enum):
    """Signal direction."""
    LONG = "long"
    SHORT = "short"
    NEUTRAL = "neutral"


class SignalTimeHorizon(str, Enum):
    """Signal time horizon."""
    INTRADAY = "intraday"
    SHORT_TERM = "1-7 days"
    MEDIUM_TERM = "1-4 weeks"
    LONG_TERM = "1-3 months"


@dataclass
class TradingSignal:
    """Structured trading signal."""
    
    commodity: str
    direction: SignalDirection
    confidence: float  # 0-1
    rationale: str
    time_horizon: SignalTimeHorizon
    generated_at: datetime
    sources: List[str]
    metadata: Dict = field(default_factory=dict)
    
    # Optional fields for execution
    target_price: Optional[float] = None
    stop_loss: Optional[float] = None
    position_size: Optional[float] = None
    
    def __post_init__(self):
        """Validate signal parameters."""
        if not 0 <= self.confidence <= 1:
            raise ValueError(f"Confidence must be 0-1, got {self.confidence}")
        if self.direction == SignalDirection.NEUTRAL and self.confidence > 0.3:
            raise ValueError("Neutral signals should have low confidence")
    
    def to_dict(self) -> Dict:
        """Convert to dictionary."""
        return {
            'commodity': self.commodity,
            'direction': self.direction.value,
            'confidence': self.confidence,
            'rationale': self.rationale,
            'time_horizon': self.time_horizon.value,
            'generated_at': self.generated_at.isoformat(),
            'sources': self.sources,
            'metadata': self.metadata,
            'target_price': self.target_price,
            'stop_loss': self.stop_loss,
            'position_size': self.position_size
        }


# Create example signal
example_signal = TradingSignal(
    commodity="WTI Crude Oil",
    direction=SignalDirection.LONG,
    confidence=0.72,
    rationale="Storage draw + strong refinery demand + OPEC cuts",
    time_horizon=SignalTimeHorizon.MEDIUM_TERM,
    generated_at=datetime.now(),
    sources=["EIA WPSR", "OPEC MOMR", "Sentiment Analysis"],
    metadata={'storage_vs_avg': -4.2, 'sentiment_score': 0.65}
)

print("Example Signal:")
print(json.dumps(example_signal.to_dict(), indent=2, default=str))

## 3. LLM-Based Signal Generator

In [ ]:
# Purpose: Build LLM-powered signal generation system
# Key Concept: Converting analysis to actionable signals

class SignalOutput(BaseModel):
    """Structured LLM signal output."""
    
    direction: Literal["long", "short", "neutral"] = Field(
        description="Trading direction recommendation"
    )
    confidence: float = Field(
        description="Confidence level (0-100)",
        ge=0, le=100
    )
    time_horizon: Literal["intraday", "1-7 days", "1-4 weeks", "1-3 months"] = Field(
        description="Expected timeframe for signal to play out"
    )
    rationale: str = Field(
        description="Clear explanation of signal logic (2-3 sentences)"
    )
    key_factors: List[str] = Field(
        description="3-5 key factors supporting this signal"
    )
    risks: List[str] = Field(
        description="2-3 key risks that could invalidate signal"
    )


class SignalGenerator:
    """Generate trading signals from fundamental analysis."""
    
    def __init__(self, api_key: Optional[str] = None):
        self.llm = ChatAnthropic(
            model="claude-3-5-sonnet-20241022",
            api_key=api_key or os.environ.get("ANTHROPIC_API_KEY"),
            temperature=0.2  # Lower temp for more consistent signals
        )
        self.parser = PydanticOutputParser(pydantic_object=SignalOutput)
        
    def generate_signal(self, 
                       commodity: str,
                       fundamental_data: Dict,
                       sentiment_data: Optional[Dict] = None,
                       price_data: Optional[Dict] = None) -> TradingSignal:
        """
        Generate trading signal from multi-source analysis.
        
        Parameters
        ----------
        commodity : str
            Commodity name
        fundamental_data : dict
            Fundamental analysis (balance, inventory, etc.)
        sentiment_data : dict, optional
            Sentiment scores from news/reports
        price_data : dict, optional
            Price levels and trends
            
        Returns
        -------
        TradingSignal
            Generated trading signal
        """
        # Build context from inputs
        context_parts = [f"COMMODITY: {commodity}\n"]
        sources = []
        
        # Add fundamental data
        context_parts.append("FUNDAMENTAL ANALYSIS:")
        for key, value in fundamental_data.items():
            context_parts.append(f"- {key}: {value}")
        sources.append("Fundamental Analysis")
        
        # Add sentiment if available
        if sentiment_data:
            context_parts.append("\nSENTIMENT ANALYSIS:")
            for key, value in sentiment_data.items():
                context_parts.append(f"- {key}: {value}")
            sources.append("Sentiment Analysis")
        
        # Add price data if available
        if price_data:
            context_parts.append("\nPRICE ACTION:")
            for key, value in price_data.items():
                context_parts.append(f"- {key}: {value}")
            sources.append("Price Analysis")
        
        context = "\n".join(context_parts)
        
        # Generate signal
        prompt = ChatPromptTemplate.from_messages([
            ("system", """You are an expert commodity trader generating actionable 
trading signals based on fundamental, sentiment, and technical analysis.

Signal Guidelines:
- LONG: Clear bullish factors (supply deficit, strong demand, bullish sentiment)
- SHORT: Clear bearish factors (oversupply, weak demand, bearish sentiment)
- NEUTRAL: Mixed signals or insufficient conviction

Confidence Calibration:
- 80-100%: Very strong, aligned signals across all sources
- 60-80%: Good signals with some conflicting factors
- 40-60%: Mixed signals, marginal edge
- <40%: Weak or conflicting signals

Be conservative - only high-conviction signals when data clearly supports direction.

{format_instructions}"""),
            ("user", "Generate a trading signal based on this analysis:\n\n{context}")
        ])
        
        chain = prompt | self.llm
        response = chain.invoke({
            "context": context,
            "format_instructions": self.parser.get_format_instructions()
        })
        
        # Parse LLM output
        signal_output = self.parser.parse(response.content)
        
        # Convert to TradingSignal
        signal = TradingSignal(
            commodity=commodity,
            direction=SignalDirection(signal_output.direction),
            confidence=signal_output.confidence / 100,  # Convert to 0-1
            rationale=signal_output.rationale,
            time_horizon=SignalTimeHorizon(signal_output.time_horizon),
            generated_at=datetime.now(),
            sources=sources,
            metadata={
                'key_factors': signal_output.key_factors,
                'risks': signal_output.risks,
                'fundamental_data': fundamental_data
            }
        )
        
        return signal


# Initialize generator
signal_generator = SignalGenerator()

print("✓ Signal generator initialized")

Test the signal generator with sample data.

In [ ]:
# Purpose: Generate sample trading signal
# Key Concept: End-to-end signal generation workflow

# Sample fundamental data (from Module 4)
crude_fundamentals = {
    'supply_demand_balance': '-7.5 million b/d (deficit)',
    'inventory_level': '439.7 million barrels',
    'vs_5yr_avg': '-4.0% (below average)',
    'weekly_change': '+8.9 million barrels',
    'days_supply': '21.1 days',
    'refinery_utilization': '90.8%',
    'production': '13.3 million b/d',
    'key_factors': 'OPEC+ cuts, strong refinery demand, low inventories'
}

# Sample sentiment data (from Module 3)
crude_sentiment = {
    'overall_sentiment': 0.42,
    'news_sentiment': 'Moderately bullish',
    'analyst_consensus': 'Buy',
    'geopolitical_risk': 'Elevated (Middle East tensions)'
}

# Sample price data
crude_price = {
    'current_price': '$72.50/bbl',
    'trend': 'Uptrend (5-day MA above 20-day MA)',
    'support': '$68.00',
    'resistance': '$78.00'
}

# Generate signal
crude_signal = signal_generator.generate_signal(
    commodity="WTI Crude Oil",
    fundamental_data=crude_fundamentals,
    sentiment_data=crude_sentiment,
    price_data=crude_price
)

# Display signal
print("=== GENERATED TRADING SIGNAL ===")
print(f"Commodity: {crude_signal.commodity}")
print(f"Direction: {crude_signal.direction.value.upper()}")
print(f"Confidence: {crude_signal.confidence:.0%}")
print(f"Time Horizon: {crude_signal.time_horizon.value}")
print(f"\nRationale:")
print(f"  {crude_signal.rationale}")
print(f"\nKey Factors:")
for factor in crude_signal.metadata['key_factors']:
    print(f"  + {factor}")
print(f"\nRisks:")
for risk in crude_signal.metadata['risks']:
    print(f"  - {risk}")
print(f"\nSources: {', '.join(crude_signal.sources)}")
print(f"Generated: {crude_signal.generated_at.strftime('%Y-%m-%d %H:%M:%S')}")

## 4. Position Sizing Based on Confidence

In [ ]:
# Purpose: Implement confidence-weighted position sizing
# Key Concept: Risk management through signal confidence

class PositionSizer:
    """Calculate position sizes based on signal confidence."""
    
    def __init__(self, 
                 max_position_size: float = 100.0,
                 min_confidence: float = 0.5,
                 risk_per_trade: float = 0.02):
        """
        Initialize position sizer.
        
        Parameters
        ----------
        max_position_size : float
            Maximum position size (% of portfolio or contracts)
        min_confidence : float
            Minimum confidence threshold to trade
        risk_per_trade : float
            Maximum risk per trade (% of portfolio)
        """
        self.max_position_size = max_position_size
        self.min_confidence = min_confidence
        self.risk_per_trade = risk_per_trade
        
    def calculate_position_size(self, signal: TradingSignal) -> float:
        """
        Calculate position size from signal confidence.
        
        Parameters
        ----------
        signal : TradingSignal
            Trading signal
            
        Returns
        -------
        float
            Position size (0 if below threshold)
        """
        # No position for neutral or low confidence
        if signal.direction == SignalDirection.NEUTRAL:
            return 0.0
        
        if signal.confidence < self.min_confidence:
            return 0.0
        
        # Scale position by confidence
        # Linear scaling: 50% conf = 0% position, 100% conf = 100% position
        confidence_scaled = (signal.confidence - self.min_confidence) / (1 - self.min_confidence)
        
        position_size = confidence_scaled * self.max_position_size
        
        return position_size
    
    def calculate_kelly_size(self, 
                            signal: TradingSignal,
                            win_rate: float = 0.55,
                            avg_win_loss_ratio: float = 1.5) -> float:
        """
        Calculate Kelly Criterion-based position size.
        
        Kelly% = (p*b - q) / b
        where p = win rate, q = loss rate, b = win/loss ratio
        
        Parameters
        ----------
        signal : TradingSignal
            Trading signal
        win_rate : float
            Historical win rate for this type of signal
        avg_win_loss_ratio : float
            Average win / average loss
            
        Returns
        -------
        float
            Kelly-adjusted position size
        """
        if signal.direction == SignalDirection.NEUTRAL or signal.confidence < self.min_confidence:
            return 0.0
        
        # Adjust win rate by signal confidence
        adjusted_win_rate = win_rate * signal.confidence
        loss_rate = 1 - adjusted_win_rate
        
        # Kelly formula
        kelly_pct = (adjusted_win_rate * avg_win_loss_ratio - loss_rate) / avg_win_loss_ratio
        kelly_pct = max(0, kelly_pct)  # No negative sizing
        
        # Use fractional Kelly (half Kelly for safety)
        fractional_kelly = 0.5
        position_size = kelly_pct * fractional_kelly * self.max_position_size
        
        # Cap at max position
        return min(position_size, self.max_position_size)


# Initialize sizer
sizer = PositionSizer(
    max_position_size=100.0,  # 100 contracts or % of portfolio
    min_confidence=0.55,      # Only trade signals >55% confidence
    risk_per_trade=0.02       # 2% risk per trade
)

# Calculate positions for our signal
linear_size = sizer.calculate_position_size(crude_signal)
kelly_size = sizer.calculate_kelly_size(crude_signal)

print("=== POSITION SIZING ===")
print(f"Signal Confidence: {crude_signal.confidence:.0%}")
print(f"Linear Sizing: {linear_size:.1f} contracts")
print(f"Kelly Sizing: {kelly_size:.1f} contracts")

# Add to signal
crude_signal.position_size = kelly_size

Visualize position sizing across different confidence levels.

In [ ]:
# Purpose: Visualize position sizing strategies
# Key Concept: Understanding risk scaling with confidence

# Generate position sizes across confidence range
confidence_levels = np.linspace(0, 1, 50)
linear_sizes = []
kelly_sizes = []

for conf in confidence_levels:
    # Create dummy signal
    dummy_signal = TradingSignal(
        commodity="Test",
        direction=SignalDirection.LONG,
        confidence=conf,
        rationale="Test",
        time_horizon=SignalTimeHorizon.MEDIUM_TERM,
        generated_at=datetime.now(),
        sources=[]
    )
    
    linear_sizes.append(sizer.calculate_position_size(dummy_signal))
    kelly_sizes.append(sizer.calculate_kelly_size(dummy_signal))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Position size vs confidence
ax1 = axes[0]
ax1.plot(confidence_levels * 100, linear_sizes, 
         label='Linear Sizing', linewidth=2, color='blue')
ax1.plot(confidence_levels * 100, kelly_sizes, 
         label='Kelly Sizing', linewidth=2, color='green')
ax1.axvline(x=sizer.min_confidence * 100, color='red', 
            linestyle='--', label='Min Confidence Threshold')
ax1.fill_between(confidence_levels * 100, 0, sizer.max_position_size,
                 where=(confidence_levels * 100) < sizer.min_confidence * 100,
                 alpha=0.2, color='red', label='No Trade Zone')
ax1.set_xlabel('Signal Confidence (%)', fontsize=12)
ax1.set_ylabel('Position Size', fontsize=12)
ax1.set_title('Position Sizing Strategies', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Risk allocation
ax2 = axes[1]
confidence_bins = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
bin_labels = ['50-60%', '60-70%', '70-80%', '80-90%', '90-100%']
avg_sizes = []

for i in range(len(confidence_bins) - 1):
    mask = (confidence_levels >= confidence_bins[i]) & (confidence_levels < confidence_bins[i+1])
    avg_sizes.append(np.mean(np.array(kelly_sizes)[mask]))

ax2.bar(bin_labels, avg_sizes, color='steelblue', alpha=0.7)
ax2.set_xlabel('Confidence Range', fontsize=12)
ax2.set_ylabel('Average Position Size', fontsize=12)
ax2.set_title('Risk Allocation by Confidence', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Signal Pipeline Architecture

In [ ]:
# Purpose: Build end-to-end signal generation pipeline
# Key Concept: Production-ready signal workflow

@dataclass
class SignalPipelineConfig:
    """Configuration for signal pipeline."""
    
    commodities: List[str]
    min_confidence: float = 0.55
    max_position_per_commodity: float = 100.0
    max_portfolio_exposure: float = 300.0
    enable_kelly_sizing: bool = True
    log_all_signals: bool = True


class SignalPipeline:
    """End-to-end signal generation pipeline."""
    
    def __init__(self, config: SignalPipelineConfig):
        self.config = config
        self.signal_generator = SignalGenerator()
        self.position_sizer = PositionSizer(
            max_position_size=config.max_position_per_commodity,
            min_confidence=config.min_confidence
        )
        self.signal_history: List[TradingSignal] = []
        
    def generate_signals(self, 
                        market_data: Dict[str, Dict]) -> List[TradingSignal]:
        """
        Generate signals for all configured commodities.
        
        Parameters
        ----------
        market_data : dict
            Dictionary mapping commodity names to their analysis data
            
        Returns
        -------
        list of TradingSignal
            Generated signals
        """
        signals = []
        
        for commodity in self.config.commodities:
            if commodity not in market_data:
                continue
            
            data = market_data[commodity]
            
            # Generate signal
            signal = self.signal_generator.generate_signal(
                commodity=commodity,
                fundamental_data=data.get('fundamentals', {}),
                sentiment_data=data.get('sentiment'),
                price_data=data.get('price')
            )
            
            # Calculate position size
            if self.config.enable_kelly_sizing:
                signal.position_size = self.position_sizer.calculate_kelly_size(signal)
            else:
                signal.position_size = self.position_sizer.calculate_position_size(signal)
            
            signals.append(signal)
            
            # Log signal
            if self.config.log_all_signals:
                self.signal_history.append(signal)
        
        # Apply portfolio-level risk controls
        signals = self._apply_portfolio_limits(signals)
        
        return signals
    
    def _apply_portfolio_limits(self, 
                               signals: List[TradingSignal]) -> List[TradingSignal]:
        """
        Apply portfolio-level position limits.
        
        Parameters
        ----------
        signals : list of TradingSignal
            Raw signals
            
        Returns
        -------
        list of TradingSignal
            Risk-adjusted signals
        """
        # Calculate total exposure
        total_exposure = sum(s.position_size for s in signals if s.position_size)
        
        # Scale down if over limit
        if total_exposure > self.config.max_portfolio_exposure:
            scale_factor = self.config.max_portfolio_exposure / total_exposure
            
            for signal in signals:
                if signal.position_size:
                    signal.position_size *= scale_factor
                    signal.metadata['scaled_for_portfolio_limit'] = True
        
        return signals
    
    def get_active_signals(self) -> List[TradingSignal]:
        """Get signals with non-zero positions."""
        return [s for s in self.signal_history 
                if s.position_size and s.position_size > 0]
    
    def get_signal_summary(self) -> pd.DataFrame:
        """Generate summary DataFrame of recent signals."""
        if not self.signal_history:
            return pd.DataFrame()
        
        summary_data = []
        for signal in self.signal_history:
            summary_data.append({
                'commodity': signal.commodity,
                'direction': signal.direction.value,
                'confidence': signal.confidence,
                'position_size': signal.position_size or 0,
                'time_horizon': signal.time_horizon.value,
                'generated_at': signal.generated_at
            })
        
        return pd.DataFrame(summary_data)


print("✓ Signal pipeline defined")

Test the pipeline with multiple commodities.

In [ ]:
# Purpose: Run signal pipeline for portfolio of commodities
# Key Concept: Multi-commodity signal generation

# Configure pipeline
config = SignalPipelineConfig(
    commodities=["WTI Crude Oil", "Natural Gas", "RBOB Gasoline"],
    min_confidence=0.55,
    max_position_per_commodity=100.0,
    max_portfolio_exposure=250.0,
    enable_kelly_sizing=True
)

pipeline = SignalPipeline(config)

# Prepare market data
market_data = {
    "WTI Crude Oil": {
        'fundamentals': crude_fundamentals,
        'sentiment': crude_sentiment,
        'price': crude_price
    },
    "Natural Gas": {
        'fundamentals': {
            'supply_demand_balance': '-22.3 Bcf/d (withdrawal season)',
            'inventory_level': '3,522 Bcf',
            'vs_5yr_avg': '+10.2% (well above average)',
            'weekly_change': '-112 Bcf (withdrawal)',
            'days_supply': '27.8 days',
            'key_factors': 'Oversupplied, warm winter forecast'
        },
        'sentiment': {
            'overall_sentiment': -0.25,
            'news_sentiment': 'Bearish',
            'weather_outlook': 'Warmer than normal'
        }
    },
    "RBOB Gasoline": {
        'fundamentals': {
            'supply_demand_balance': '+1.4 million b/d (surplus)',
            'inventory_level': '225 million barrels',
            'vs_5yr_avg': '+2.5% (slightly above average)',
            'crack_spread': '$18.50/bbl (healthy margins)',
            'key_factors': 'Strong refinery runs, seasonal build'
        }
    }
}

# Generate signals
signals = pipeline.generate_signals(market_data)

# Display results
print("=== SIGNAL GENERATION COMPLETE ===")
print(f"\nGenerated {len(signals)} signals")
print(f"Active signals: {len([s for s in signals if s.position_size and s.position_size > 0])}")

for signal in signals:
    print(f"\n{signal.commodity}:")
    print(f"  Direction: {signal.direction.value.upper()}")
    print(f"  Confidence: {signal.confidence:.0%}")
    print(f"  Position Size: {signal.position_size:.1f}")
    print(f"  Rationale: {signal.rationale[:100]}...")

# Get summary
summary_df = pipeline.get_signal_summary()
print("\n=== SIGNAL SUMMARY ===")
print(summary_df)

## 6. Signal Monitoring Dashboard

In [ ]:
# Purpose: Visualize signal portfolio
# Key Concept: Portfolio-level signal monitoring

def create_signal_dashboard(signals: List[TradingSignal]):
    """
    Create signal monitoring dashboard.
    
    Parameters
    ----------
    signals : list of TradingSignal
        Current signals
    """
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Position allocation
    ax1 = axes[0, 0]
    active_signals = [s for s in signals if s.position_size and s.position_size > 0]
    commodities = [s.commodity for s in active_signals]
    positions = [s.position_size for s in active_signals]
    
    if active_signals:
        ax1.pie(positions, labels=commodities, autopct='%1.1f%%', startangle=90)
        ax1.set_title('Position Allocation', fontsize=12, fontweight='bold')
    else:
        ax1.text(0.5, 0.5, 'No Active Positions', ha='center', va='center')
        ax1.set_title('Position Allocation', fontsize=12, fontweight='bold')
    
    # Plot 2: Direction distribution
    ax2 = axes[0, 1]
    direction_counts = {}
    for s in signals:
        direction_counts[s.direction.value] = direction_counts.get(s.direction.value, 0) + 1
    
    colors_map = {'long': 'green', 'short': 'red', 'neutral': 'gray'}
    ax2.bar(direction_counts.keys(), direction_counts.values(),
           color=[colors_map.get(k, 'blue') for k in direction_counts.keys()],
           alpha=0.7)
    ax2.set_ylabel('Count', fontsize=12)
    ax2.set_title('Signal Direction Distribution', fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    
    # Plot 3: Confidence vs Position Size
    ax3 = axes[1, 0]
    confidences = [s.confidence * 100 for s in signals]
    position_sizes = [s.position_size or 0 for s in signals]
    colors = ['green' if s.direction == SignalDirection.LONG 
              else 'red' if s.direction == SignalDirection.SHORT 
              else 'gray' for s in signals]
    
    ax3.scatter(confidences, position_sizes, c=colors, s=100, alpha=0.6)
    for i, signal in enumerate(signals):
        ax3.annotate(signal.commodity.split()[0], 
                    (confidences[i], position_sizes[i]),
                    fontsize=8, ha='right')
    ax3.set_xlabel('Confidence (%)', fontsize=12)
    ax3.set_ylabel('Position Size', fontsize=12)
    ax3.set_title('Confidence vs Position Size', fontsize=12, fontweight='bold')
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Signal summary table
    ax4 = axes[1, 1]
    ax4.axis('off')
    
    total_position = sum(s.position_size or 0 for s in signals)
    avg_confidence = np.mean([s.confidence for s in signals])
    
    summary_text = f"""
SIGNAL PORTFOLIO SUMMARY
{'='*40}

Total Signals: {len(signals)}
Active Positions: {len(active_signals)}
Total Exposure: {total_position:.1f}
Average Confidence: {avg_confidence:.1%}

BREAKDOWN:
Long: {sum(1 for s in signals if s.direction == SignalDirection.LONG)}
Short: {sum(1 for s in signals if s.direction == SignalDirection.SHORT)}
Neutral: {sum(1 for s in signals if s.direction == SignalDirection.NEUTRAL)}

TOP CONVICTION:
    """
    
    # Add top signal
    sorted_signals = sorted(signals, key=lambda s: s.confidence, reverse=True)
    if sorted_signals:
        top = sorted_signals[0]
        summary_text += f"""
{top.commodity}
  {top.direction.value.upper()} @ {top.confidence:.0%}
  Size: {top.position_size or 0:.1f}
        """
    
    ax4.text(0.1, 0.9, summary_text, transform=ax4.transAxes,
            fontsize=10, verticalalignment='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    
    plt.suptitle('Signal Monitoring Dashboard', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()


# Generate dashboard
create_signal_dashboard(signals)

## Exercise 5.1: Build Your Own Signal Generator

In [ ]:
# Exercise: Create signals for different commodities
#
# Task:
# 1. Choose 3 different commodities (e.g., gold, wheat, copper)
# 2. Create fundamental data for each
# 3. Generate signals using the pipeline
# 4. Analyze the resulting portfolio allocation

# YOUR CODE HERE
# ---------------

# Step 1: Define your commodities and data
my_market_data = {
    # Add your commodity data here
}

# Step 2: Generate signals
my_signals = None  # Use pipeline.generate_signals()

# Step 3: Analyze results
# Create dashboard and analyze allocation

## Summary

### Key Takeaways

1. **Structured Signals**: Trading signals require clear direction, confidence, rationale, and timeframe
2. **LLM Integration**: LLMs convert multi-source analysis into actionable signals
3. **Confidence-Based Sizing**: Position size scales with signal confidence for risk management
4. **Pipeline Architecture**: Production systems need data ingestion → analysis → signal → sizing → risk control
5. **Portfolio Management**: Apply limits at both position and portfolio levels

### What's Next

In **Module 5.2: Backtest Analysis**, we'll:
- Test signal quality on historical data
- Measure precision, profit factor, and Sharpe ratio
- Calibrate confidence scores
- Optimize signal thresholds

### Additional Resources

- "Quantitative Trading" by Ernie Chan
- "Evidence-Based Technical Analysis" by David Aronson
- Kelly Criterion: "Fortune's Formula" by William Poundstone
- Position Sizing strategies: Van Tharp's work

---

**Practice Exercise**: Run the signal pipeline daily for a week. Track how signals evolve as new fundamental data arrives. Analyze signal persistence and confidence changes.